<a href="https://colab.research.google.com/github/AchsahGraceW/SpectralCodeAstrobit/blob/main/astrobit_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

# The 3 zip files in your Google Drive
zip_files = ['dev_pack.zip', 'private_pack.zip', 'train_pack.zip']

# Target destination folder inside Colab
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)

# Unzip each file
for zip_name in zip_files:
    zip_path = os.path.join('/content/drive/MyDrive', zip_name)

    if os.path.exists(zip_path):
        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Successfully extracted {zip_name}!")
    else:
        print(f"Error: {zip_name} was not found in your main Google Drive folder!")

Extracting dev_pack.zip...
Successfully extracted dev_pack.zip!
Extracting private_pack.zip...
Successfully extracted private_pack.zip!
Extracting train_pack.zip...
Successfully extracted train_pack.zip!


In [5]:
import glob
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import BoxLeastSquares
warnings.filterwarnings("ignore")

In [6]:
TRAIN_DIR = "/content/dataset/train"
DEV_DIR = "/content/dataset/dev"
PRIVATE_DIR = "/content/dataset/private"

print("train  ", len(glob.glob(f"{TRAIN_DIR}/*.parquet")), "stars")
print("dev    ", len(glob.glob(f"{DEV_DIR}/*.parquet")), "stars")
print("private", len(glob.glob(f"{PRIVATE_DIR}/*.parquet")), "stars")

train   269 stars
dev     89 stars
private 87 stars


# Starter Pipeline — Transit Detection in Kepler Photometry

This notebook runs **end to end**: it loads a light curve, cleans it, searches for a
periodic transit signal, and writes a valid submission file.

It is deliberately simple. It is a **floor to beat, not a solution to tweak.**
Every stage below is the most obvious thing that works, and every stage can be
improved substantially. Your job is to replace them.

## The pipeline

```
raw SAP flux
  -> mask bad cadences        (quality flags)
  -> normalise per quarter    (remove inter-quarter jumps)
  -> detrend                  (remove stellar variability + drift)
  -> BLS period search        (find the transit)
  -> threshold + report       (decide, and measure)
```

## Read this before you start

**A naive period grid will find nothing.** This is the single biggest trap in
this problem. A full-resolution BLS search to 400 days needs ~14 million trial
periods — roughly 5 hours *per star*. If you instead use a coarse fixed grid,
your trial periods land too far from the true one, the fold smears the transit
away, and you detect nothing at all — even for signals that are obvious by eye.

The pipeline below uses a **coarse-to-fine** search: a wide cheap sweep to
localise candidates, then a fine sweep around each peak. That makes the search
tractable. Understand this part before changing it.

**Develop on a handful of stars.** Run the full set only when you have something
worth testing.

In [ ]:
!pip install -q astropy pyarrow pandas numpy matplotlib

In [ ]:
import glob
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares

warnings.filterwarnings("ignore")

# point these at wherever you unzipped the packs
TRAIN_DIR = "train"
DEV_DIR = "dev"
PRIVATE_DIR = "private"

print("train  ", len(glob.glob(f"{TRAIN_DIR}/*.parquet")), "stars")
print("dev    ", len(glob.glob(f"{DEV_DIR}/*.parquet")), "stars")
print("private", len(glob.glob(f"{PRIVATE_DIR}/*.parquet")), "stars")

## 1. What a light curve looks like

Each file is a single star's brightness over ~4 years, sampled every 29.4 minutes.

| column | meaning |
|---|---|
| `time` | days |
| `flux` | raw SAP flux — **not** detrended |
| `flux_err` | uncertainty per measurement |
| `quality` | bitmask; 0 means good |
| `quarter` | which ~90-day observing quarter |

The flux is **raw**. Removing instrumental and stellar signals without destroying
the transit is a core part of the problem, not a preprocessing detail.

In [ ]:
files = sorted(glob.glob(f"{TRAIN_DIR}/*.parquet"))
df = pd.read_parquet(files[0])

print(df.shape)
print(df.head())
print(f"\nbaseline: {df.time.max() - df.time.min():.0f} days")
print(f"quarters: {sorted(df.quarter.unique())}")
print(f"bad cadences: {(df.quality != 0).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
m = df.quality == 0
ax.plot(df.time[m], df.flux[m], "k.", ms=0.5, alpha=0.5)
ax.set_xlabel("time (days)")
ax.set_ylabel("SAP flux")
ax.set_title(os.path.basename(files[0]))
plt.tight_layout()
plt.show()

# The step changes are quarter boundaries: the spacecraft rolled every ~90 days,
# putting the star on a different detector. The slow wiggles within a quarter are
# a mix of stellar variability and instrumental drift. A transit is a dip of
# 0.01%-0.2% lasting a few hours. You cannot see it here. That is the problem.

## 2. Cleaning

Three steps, each of which you should question:

**Quality mask.** Drop cadences flagged by the mission — cosmic rays, thruster
firings, safe-mode recovery. Dropping everything non-zero is blunt; the bits mean
different things and some are harmless.

**Per-quarter normalisation.** Divide each quarter by its median to remove the
jumps. Crude: it assumes flux within a quarter is stationary, which it is not.

**Detrending.** Divide by a running median. The window length is the critical
parameter — too long and variability survives, too short and you eat the transit
itself. A 1-day window is safe for transits lasting a few hours, and wrong for
long-duration transits of long-period planets.

Better approaches exist: spline or Savitzky-Golay filtering, Gaussian processes,
PCA against other stars, iterative masking of in-transit points. This is
the highest-leverage part of the pipeline — in testing, proper detrending took
recovered transit depth from 33% of true to 90%.

In [7]:
DETREND_WINDOW_DAYS = 1.0

def clean(df, window_days=DETREND_WINDOW_DAYS):
    m = (df.quality.values == 0) & np.isfinite(df.flux.values)
    t = df.time.values[m]
    f = df.flux.values[m].astype(float)
    q = df.quarter.values[m]
    if len(t) < 1000:
        return None, None

    for qq in np.unique(q):
        s = q == qq
        med = np.median(f[s])
        f[s] = f[s] / med if med > 0 else 1.0

    cadence = np.median(np.diff(t))
    k = max(5, int(window_days / cadence) | 1)
    trend = pd.Series(f).rolling(k, center=True, min_periods=k // 3).median().values

    ok = np.isfinite(trend) & (trend > 0)
    return t[ok], f[ok] / trend[ok]

In [ ]:
DETREND_WINDOW_DAYS = 1.0


def clean(df, window_days=DETREND_WINDOW_DAYS):
    """Raw light curve -> (time, normalised flux) ready for a period search."""
    m = (df.quality.values == 0) & np.isfinite(df.flux.values)
    t = df.time.values[m]
    f = df.flux.values[m].astype(float)
    q = df.quarter.values[m]
    if len(t) < 1000:
        return None, None

    # normalise each quarter to its own median
    for qq in np.unique(q):
        s = q == qq
        med = np.median(f[s])
        f[s] = f[s] / med if med > 0 else 1.0

    # centred running median as the trend estimate
    cadence = np.median(np.diff(t))
    k = max(5, int(window_days / cadence) | 1)  # odd window
    trend = pd.Series(f).rolling(k, center=True, min_periods=k // 3).median().values

    ok = np.isfinite(trend) & (trend > 0)
    return t[ok], f[ok] / trend[ok]


t, f = clean(df)
print(f"{len(t)} cadences, scatter {np.std(f) * 1e6:.0f} ppm")

fig, ax = plt.subplots(figsize=(15, 3))
ax.plot(t, f, "k.", ms=0.5, alpha=0.5)
ax.set_ylim(np.percentile(f, 0.1), np.percentile(f, 99.9))
ax.set_xlabel("time (days)")
ax.set_ylabel("normalised flux")
plt.tight_layout()
plt.show()

# Compare that scatter to the transit depths you are hunting: 100-2000 ppm.
# The shallowest signals are well below the per-point noise. They only emerge
# once you fold many transits together at the correct period.

In [ ]:
from scipy.signal import savgol_filter

def clean_v2(df, window_days=1.0):
    """Improved cleaning: Savitzky-Golay detrending instead of rolling median.
    This preserves sharp features (like transits) better while still
    removing smooth stellar variability."""
    m = (df.quality.values == 0) & np.isfinite(df.flux.values)
    t = df.time.values[m]
    f = df.flux.values[m].astype(float)
    q = df.quarter.values[m]
    if len(t) < 1000:
        return None, None

    # normalise each quarter to its own median (same as before)
    for qq in np.unique(q):
        s = q == qq
        med = np.median(f[s])
        f[s] = f[s] / med if med > 0 else 1.0

    # Savitzky-Golay detrend per quarter (handles quarter-specific drift
    # better than one global rolling median)
    cadence = np.median(np.diff(t))
    window_len = int(window_days / cadence)
    if window_len % 2 == 0:
        window_len += 1  # must be odd
    window_len = max(window_len, 5)

    trend = np.ones_like(f)
    for qq in np.unique(q):
        s = q == qq
        if s.sum() > window_len:
            trend[s] = savgol_filter(f[s], window_length=min(window_len, s.sum() - (1 - s.sum() % 2)), polyorder=2)
        else:
            trend[s] = np.median(f[s])

    ok = np.isfinite(trend) & (trend > 0)
    return t[ok], f[ok] / trend[ok]

In [ ]:
t2, f2 = clean_v2(pd.read_parquet(f"{TRAIN_DIR}/KIC_9933041.parquet"))  # the shallow 102ppm one that failed
print(f"{len(t2)} cadences, scatter {np.std(f2) * 1e6:.0f} ppm")

result2 = search_smart(t2, f2, verbose=True)
print(result2)

51107 cadences, scatter 327 ppm
  P=  48.846 ->  48.8475  SDE  14.9  depth    518ppm  n_transits    30  plausibility    318.8
  P=  42.771 ->  42.7700  SDE  11.0  depth    407ppm  n_transits    34  plausibility    234.9
  P=  31.210 ->  31.2094  SDE   9.9  depth    313ppm  n_transits    47  plausibility    219.5
  P=  24.424 ->  24.4245  SDE   8.6  depth    258ppm  n_transits    60  plausibility    196.8
  P=  38.494 ->  38.4928  SDE   8.6  depth    346ppm  n_transits    38  plausibility    184.1
  P=  27.495 ->  27.4940  SDE   6.5  depth    255ppm  n_transits    53  plausibility    143.5
  P=  16.265 ->  16.2652  SDE   5.3  depth    181ppm  n_transits    90  plausibility    124.7
  P=  14.043 ->  14.0435  SDE   4.8  depth    170ppm  n_transits   104  plausibility    114.3
{'period': 48.84753503211658, 'depth_ppm': 517.7867881574882, 'duration_hours': 4.800000000000001, 't0': 132.56773468948785, 'sde': 14.868891228226717, 'plausibility': np.float64(318.8334791302004)}


In [ ]:
t3, f3 = clean_v2(pd.read_parquet(f"{TRAIN_DIR}/KIC_4935189.parquet"))  # the easy 1977ppm one
result3 = search_smart(t3, f3, verbose=False)
err3 = abs(result3["period"] - 9.3204) / 9.3204
print(f"recovered P={result3['period']:.4f}d  error={err3:.2%}  {'✅' if err3 < 0.02 else '❌'}")

recovered P=9.3206d  error=0.00%  ✅


In [ ]:
# Compare old vs new detrending on the medium star
t_old, f_old = clean(pd.read_parquet(f"{TRAIN_DIR}/KIC_7213876.parquet"))
t_new, f_new = clean_v2(pd.read_parquet(f"{TRAIN_DIR}/KIC_7213876.parquet"))

print(f"old scatter: {np.std(f_old)*1e6:.0f} ppm")
print(f"new scatter: {np.std(f_new)*1e6:.0f} ppm")

result_new = search_smart(t_new, f_new, verbose=False)
err_new = abs(result_new["period"] - 61.7771) / 61.7771
print(f"new recovered P={result_new['period']:.4f}d  error={err_new:.2%}  {'✅' if err_new < 0.02 else '❌'}")

old scatter: 498 ppm
new scatter: 509 ppm
new recovered P=45.8803d  error=25.73%  ❌


## 3. The period search

Box Least Squares folds the light curve at many trial periods and looks for a
box-shaped dip.

**Why the grid matters so much.** To keep a transit aligned across the whole
4-year baseline, a trial period must be accurate to roughly
`duration / n_transits`. For a 3-hour transit at 35 days, that is about
0.003 days. A 3000-point grid spanning 3–400 days gives spacing of ~0.04 days
near 35 days — more than ten times too coarse. Every trial misses, the folded
transit smears across phase, and the signal vanishes.

The required resolution scales as roughly `P^2 / (baseline x duration)`, so the
cost of a complete search grows steeply with maximum period. Searching to 400
days at full resolution is ~14 million trials.

**Coarse-to-fine.** Sweep a cheap grid to find regions with elevated power, then
re-search finely only around those peaks. Roughly 100x cheaper for most of the
sensitivity. This is what the function below does, and it is the part worth
understanding before you change anything else.

In [8]:
PERIOD_MIN = 3.0
PERIOD_MAX = 400.0
N_COARSE = 20000
N_PEAKS = 8
N_FINE = 600
DURATIONS = np.array([0.05, 0.1, 0.2, 0.4, 0.8])

def sde(power, i):
    med = np.nanmedian(power)
    mad = np.nanmedian(np.abs(power - med))
    return float((power[i] - med) / (1.4826 * mad)) if mad > 0 else 0.0

def search(t, f, verbose=False):
    bls = BoxLeastSquares(t, f)
    baseline = t.max() - t.min()
    pmax = min(PERIOD_MAX, baseline / 3.0)

    coarse = np.exp(np.linspace(np.log(PERIOD_MIN), np.log(pmax), N_COARSE))
    res = bls.power(coarse, DURATIONS, objective="likelihood")
    power = np.asarray(res.power)

    order = np.argsort(power)[::-1]
    peaks, used = [], np.zeros(len(coarse), bool)
    for i in order:
        if used[i]:
            continue
        peaks.append(i)
        lo = np.searchsorted(coarse, coarse[i] * 0.9)
        hi = np.searchsorted(coarse, coarse[i] * 1.1)
        used[lo:hi] = True
        if len(peaks) >= N_PEAKS:
            break

    best = None
    for i in peaks:
        p0 = coarse[i]
        width = 0.02 * p0
        fine = np.linspace(p0 - width, p0 + width, N_FINE)
        fine = fine[fine > PERIOD_MIN]
        if len(fine) < 10:
            continue

        r = bls.power(fine, DURATIONS, objective="likelihood")
        p = np.asarray(r.power)
        j = int(np.nanargmax(p))
        score = sde(power, i)

        if best is None or score > best["sde"]:
            best = {
                "period": float(r.period[j]),
                "depth_ppm": float(r.depth[j] * 1e6),
                "duration_hours": float(r.duration[j] * 24),
                "t0": float(r.transit_time[j]),
                "sde": score,
            }
        if verbose:
            print(f"  peak P={p0:8.3f} -> refined {r.period[j]:8.4f}  SDE {score:5.1f}")

    return best or {"period": np.nan, "depth_ppm": np.nan,
                    "duration_hours": np.nan, "t0": np.nan, "sde": 0.0}

In [11]:
def search_smart(t, f, verbose=False):
    bls = BoxLeastSquares(t, f)
    baseline = t.max() - t.min()
    pmax = min(PERIOD_MAX, baseline / 3.0)

    coarse = np.exp(np.linspace(np.log(PERIOD_MIN), np.log(pmax), N_COARSE))
    res = bls.power(coarse, DURATIONS, objective="likelihood")
    power = np.asarray(res.power)

    order = np.argsort(power)[::-1]
    peaks, used = [], np.zeros(len(coarse), bool)
    for i in order:
        if used[i]:
            continue
        peaks.append(i)
        lo = np.searchsorted(coarse, coarse[i] * 0.9)
        hi = np.searchsorted(coarse, coarse[i] * 1.1)
        used[lo:hi] = True
        if len(peaks) >= N_PEAKS:
            break

    candidates = []
    for i in peaks:
        p0 = coarse[i]
        width = 0.02 * p0
        fine = np.linspace(p0 - width, p0 + width, N_FINE)
        fine = fine[fine > PERIOD_MIN]
        if len(fine) < 10:
            continue

        r = bls.power(fine, DURATIONS, objective="likelihood")
        p = np.asarray(r.power)
        j = int(np.nanargmax(p))
        score = sde(power, i)

        n_transits = baseline / r.period[j]
        depth = r.depth[j] * 1e6

        plausibility = score * np.log1p(depth) * np.log1p(n_transits)

        candidates.append({
            "period": float(r.period[j]),
            "depth_ppm": float(depth),
            "duration_hours": float(r.duration[j] * 24),
            "t0": float(r.transit_time[j]),
            "sde": score,
            "plausibility": plausibility,
        })
        if verbose:
            print(f"  P={p0:8.3f} -> {r.period[j]:8.4f}  SDE {score:5.1f}  depth {depth:6.0f}ppm  n_transits {n_transits:5.0f}  plausibility {plausibility:8.1f}")

    if not candidates:
        return {"period": np.nan, "depth_ppm": np.nan, "duration_hours": np.nan, "t0": np.nan, "sde": 0.0, "plausibility": 0.0}

    best_candidate = candidates[0]
    for c in candidates:
        if c["plausibility"] > best_candidate["plausibility"]:
            best_candidate = c

    return best_candidate



In [12]:
truth = pd.read_csv("train_pack.zip".replace(".zip", "") + "_labels.csv") if False else None  # placeholder, ignore

truth = pd.read_csv("/content/dataset/train_truth.csv")
row = truth[truth.kepid == 4935189].iloc[0]

t, f = clean(pd.read_parquet(f"{TRAIN_DIR}/KIC_4935189.parquet"))
found = search_smart(t, f, verbose=True)

err = abs(found["period"] - row.period_days) / row.period_days
print(f"\ntrue P={row.period_days:.4f}d  recovered P={found['period']:.4f}d  error={err:.2%}  {'✅' if err < 0.02 else '❌'}")

  P= 325.693 -> 325.7254  SDE  25.5  depth   7726ppm  n_transits     5  plausibility    390.0
  P= 372.695 -> 372.6324  SDE  18.8  depth   9327ppm  n_transits     4  plausibility    274.0
  P= 217.783 -> 217.7753  SDE  13.4  depth   5890ppm  n_transits     7  plausibility    238.9
  P= 181.584 -> 181.5901  SDE  10.7  depth   4354ppm  n_transits     8  plausibility    198.8
  P= 290.315 -> 290.3247  SDE  10.3  depth   6369ppm  n_transits     5  plausibility    162.5
  P= 145.199 -> 145.1843  SDE  10.0  depth   4621ppm  n_transits    10  plausibility    203.1
  P= 245.640 -> 245.6155  SDE   9.1  depth   4222ppm  n_transits     6  plausibility    147.3
  P= 162.853 -> 162.8366  SDE   7.8  depth   3212ppm  n_transits     9  plausibility    145.5

true P=9.3204d  recovered P=325.7254d  error=3394.74%  ❌


In [ ]:
PERIOD_MIN = 3.0
PERIOD_MAX = 400.0
N_COARSE = 20000       # raise for better sensitivity, at linear cost
N_PEAKS = 8            # how many coarse peaks to refine
N_FINE = 600           # trials per refinement window
DURATIONS = np.array([0.05, 0.1, 0.2, 0.4, 0.8])  # days


def sde(power, i):
    """Signal Detection Efficiency: peak height above the periodogram floor,
    in robust standard deviations. MAD is used so a strong peak does not
    inflate its own noise estimate."""
    med = np.nanmedian(power)
    mad = np.nanmedian(np.abs(power - med))
    return float((power[i] - med) / (1.4826 * mad)) if mad > 0 else 0.0


def search(t, f, verbose=False):
    """Coarse-to-fine BLS. Returns the best candidate."""
    bls = BoxLeastSquares(t, f)
    baseline = t.max() - t.min()
    pmax = min(PERIOD_MAX, baseline / 3.0)   # insist on >= 3 transits

    # --- coarse sweep, log-spaced ---
    coarse = np.exp(np.linspace(np.log(PERIOD_MIN), np.log(pmax), N_COARSE))
    res = bls.power(coarse, DURATIONS, objective="likelihood")
    power = np.asarray(res.power)

    # --- pick well-separated peaks ---
    order = np.argsort(power)[::-1]
    peaks, used = [], np.zeros(len(coarse), bool)
    for i in order:
        if used[i]:
            continue
        peaks.append(i)
        lo = np.searchsorted(coarse, coarse[i] * 0.9)
        hi = np.searchsorted(coarse, coarse[i] * 1.1)
        used[lo:hi] = True
        if len(peaks) >= N_PEAKS:
            break

    # --- refine around each peak ---
    best = None
    for i in peaks:
        p0 = coarse[i]
        width = 0.02 * p0                         # +/- 2%
        fine = np.linspace(p0 - width, p0 + width, N_FINE)
        fine = fine[fine > PERIOD_MIN]
        if len(fine) < 10:
            continue

        r = bls.power(fine, DURATIONS, objective="likelihood")
        p = np.asarray(r.power)
        j = int(np.nanargmax(p))
        score = sde(power, i)                     # rank on the coarse periodogram

        if best is None or score > best["sde"]:
            best = {
                "period": float(r.period[j]),
                "depth_ppm": float(r.depth[j] * 1e6),
                "duration_hours": float(r.duration[j] * 24),
                "t0": float(r.transit_time[j]),
                "sde": score,
            }
        if verbose:
            print(f"  peak P={p0:8.3f} -> refined {r.period[j]:8.4f}  SDE {score:5.1f}")

    return best or {"period": np.nan, "depth_ppm": np.nan,
                    "duration_hours": np.nan, "t0": np.nan, "sde": 0.0}

## 4. Does it work?

`train_truth.csv` lists the injected signals with their true parameters. Take the
deepest one and check the pipeline recovers it. **Always verify on a known signal
before trusting a null result on unknown data** — if we had not done this, we
would have concluded the data was empty when the search grid was simply wrong.

In [ ]:
truth = pd.read_csv("train_truth.csv")
row = truth[truth.injected == 1].sort_values("depth_ppm", ascending=False).iloc[0]

print(f"KIC {int(row.kepid)}")
print(f"  true period   {row.period_days:.4f} d")
print(f"  true depth    {row.depth_ppm:.0f} ppm")
print(f"  true duration {row.duration_hours:.2f} h")
print(f"  transits      {int(row.n_transits)}\n")

t, f = clean(pd.read_parquet(f"{TRAIN_DIR}/KIC_{int(row.kepid)}.parquet"))
found = search(t, f, verbose=True)

err = abs(found["period"] - row.period_days) / row.period_days
print(f"\nrecovered  P={found['period']:.4f} d  ({err:.2%} error)")
print(f"           depth={found['depth_ppm']:.0f} ppm  SDE={found['sde']:.1f}")

In [ ]:
def plot_folded(t, f, period, t0, depth_ppm=None, bins=120):
    """Fold on a period and bin. If the period is right, a dip appears at 0."""
    ph = ((t - t0) / period) % 1.0
    ph = np.where(ph > 0.5, ph - 1, ph)
    o = np.argsort(ph)
    ph, ff = ph[o], f[o]

    edges = np.linspace(-0.5, 0.5, bins + 1)
    idx = np.digitize(ph, edges) - 1
    bx = np.array([np.median(ph[idx == i]) if (idx == i).sum() else np.nan
                   for i in range(bins)])
    by = np.array([np.median(ff[idx == i]) if (idx == i).sum() else np.nan
                   for i in range(bins)])

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(ph, ff, ".", color="0.8", ms=1, alpha=0.3)
    ax.plot(bx, by, "o-", color="crimson", ms=3, lw=1)
    if depth_ppm:
        ax.axhline(1 - depth_ppm / 1e6, ls="--", color="steelblue",
                   label=f"true depth {depth_ppm:.0f} ppm")
        ax.legend()
    ax.set_xlim(-0.1, 0.1)
    ax.set_ylim(np.nanpercentile(by, 0) - 3e-4, np.nanpercentile(by, 100) + 2e-4)
    ax.set_xlabel("phase")
    ax.set_ylabel("normalised flux")
    ax.set_title(f"folded at P = {period:.4f} d")
    plt.tight_layout()
    plt.show()


plot_folded(t, f, found["period"], found["t0"], row.depth_ppm)

# Try folding on a deliberately wrong period, e.g. found["period"] * 1.01.
# The dip disappears completely. That is how sharp the grid requirement is.

In [ ]:
PERIOD_MIN = 3.0
PERIOD_MAX = 50.0
found_narrow = search(t, f, verbose=True)
print(found_narrow)

  peak P=   3.609 -> refined   3.6092  SDE  21.0
  peak P=   9.320 -> refined   9.3206  SDE  20.7
  peak P=  46.525 -> refined  46.5237  SDE  12.3
  peak P=   7.219 -> refined   7.2188  SDE  10.4
  peak P=   4.660 -> refined   4.6601  SDE   9.8
  peak P=  18.640 -> refined  18.6406  SDE   9.3
  peak P=  36.316 -> refined  36.3149  SDE   8.5
  peak P=  28.676 -> refined  28.6771  SDE   6.7
{'period': 3.60923559613287, 'depth_ppm': 297.985117837375, 'duration_hours': 19.200000000000003, 't0': 121.87391220220975, 'sde': 20.972255291585352}


In [ ]:
def search_smart(t, f, verbose=False):
    """Like search(), but considers all refined candidates and picks the
    most plausible one using both SDE and signal strength — not just
    whichever has the single highest SDE."""
    bls = BoxLeastSquares(t, f)
    baseline = t.max() - t.min()
    pmax = min(PERIOD_MAX, baseline / 3.0)

    coarse = np.exp(np.linspace(np.log(PERIOD_MIN), np.log(pmax), N_COARSE))
    res = bls.power(coarse, DURATIONS, objective="likelihood")
    power = np.asarray(res.power)

    order = np.argsort(power)[::-1]
    peaks, used = [], np.zeros(len(coarse), bool)
    for i in order:
        if used[i]:
            continue
        peaks.append(i)
        lo = np.searchsorted(coarse, coarse[i] * 0.9)
        hi = np.searchsorted(coarse, coarse[i] * 1.1)
        used[lo:hi] = True
        if len(peaks) >= N_PEAKS:
            break

    candidates = []
    for i in peaks:
        p0 = coarse[i]
        width = 0.02 * p0
        fine = np.linspace(p0 - width, p0 + width, N_FINE)
        fine = fine[fine > PERIOD_MIN]
        if len(fine) < 10:
            continue

        r = bls.power(fine, DURATIONS, objective="likelihood")
        p = np.asarray(r.power)
        j = int(np.nanargmax(p))
        score = sde(power, i)

        n_transits = baseline / r.period[j]
        depth = r.depth[j] * 1e6

        # A believable candidate: reasonably deep AND has enough
        # transits to be confidently detected. Penalize very shallow,
        # low-transit-count candidates even if their raw SDE is high.
        plausibility = score * np.log1p(depth) * np.log1p(n_transits)

        candidates.append({
            "period": float(r.period[j]),
            "depth_ppm": float(depth),
            "duration_hours": float(r.duration[j] * 24),
            "t0": float(r.transit_time[j]),
            "sde": score,
            "plausibility": plausibility,
        })
        if verbose:
            print(f"  P={p0:8.3f} -> {r.period[j]:8.4f}  SDE {score:5.1f}  "
                  f"depth {depth:6.0f}ppm  n_transits {n_transits:5.0f}  "
                  f"plausibility {plausibility:8.1f}")

    if not candidates:
        return {"period": np.nan, "depth_ppm": np.nan,
                "duration_hours": np.nan, "t0": np.nan, "sde": 0.0}

    best = max(candidates, key=lambda c: c["plausibility"])
    return best

In [ ]:
found_smart = search_smart(t, f, verbose=True)
print(found_smart)

  P=   3.609 ->   3.6092  SDE  21.0  depth    298ppm  n_transits   407  plausibility    718.8
  P=   9.320 ->   9.3206  SDE  20.7  depth   1317ppm  n_transits   158  plausibility    753.1
  P=  46.525 ->  46.5237  SDE  12.3  depth   1257ppm  n_transits    32  plausibility    306.7
  P=   7.219 ->   7.2188  SDE  10.4  depth    303ppm  n_transits   204  plausibility    316.1
  P=   4.660 ->   4.6601  SDE   9.8  depth    671ppm  n_transits   316  plausibility    368.5
  P=  18.640 ->  18.6406  SDE   9.3  depth   1403ppm  n_transits    79  plausibility    296.6
  P=  36.316 ->  36.3149  SDE   8.5  depth    969ppm  n_transits    40  plausibility    216.9
  P=  28.676 ->  28.6771  SDE   6.7  depth    702ppm  n_transits    51  plausibility    173.5
{'period': 9.320586294627518, 'depth_ppm': 1317.0905136418414, 'duration_hours': 2.4000000000000004, 't0': 121.81891220220976, 'sde': 20.686589413599503, 'plausibility': np.float64(753.0720423419652)}


In [ ]:
import random

random.seed(42)  # so results are reproducible

truth = pd.read_csv("train_truth.csv")

# Pick a range of known planets: one deep (easy), one medium, one shallow (hard)
planets = truth[truth.injected == 1].sort_values("depth_ppm", ascending=False)
test_planets = pd.concat([
    planets.iloc[[0]],                          # deepest / easiest
    planets.iloc[[len(planets)//2]],             # medium
    planets.iloc[[-1]],                          # shallowest / hardest
])

# Pick one known-empty star for comparison
empty_star = truth[truth.injected == 0].iloc[0]

print("=== Testing on known PLANET stars ===\n")
for _, row in test_planets.iterrows():
    path = f"{TRAIN_DIR}/KIC_{int(row.kepid)}.parquet"
    t_test, f_test = clean(pd.read_parquet(path))
    if t_test is None:
        print(f"KIC {int(row.kepid)}: not enough data, skipping\n")
        continue

    result = search_smart(t_test, f_test, verbose=False)
    err = abs(result["period"] - row.period_days) / row.period_days

    print(f"KIC {int(row.kepid)}")
    print(f"  true:      P={row.period_days:.4f}d  depth={row.depth_ppm:.0f}ppm")
    print(f"  recovered: P={result['period']:.4f}d  depth={result['depth_ppm']:.0f}ppm")
    print(f"  period error: {err:.2%}  {'✅ MATCH' if err < 0.02 else '❌ MISMATCH'}\n")

print("=== Testing on a known EMPTY star (should find nothing convincing) ===\n")
path = f"{TRAIN_DIR}/KIC_{int(empty_star.kepid)}.parquet"
t_test, f_test = clean(pd.read_parquet(path))
result = search_smart(t_test, f_test, verbose=False)
print(f"KIC {int(empty_star.kepid)} (no real planet)")
print(f"  recovered: P={result['period']:.4f}d  depth={result['depth_ppm']:.0f}ppm  SDE={result['sde']:.1f}")
print(f"  (low SDE / low plausibility here is a good sign)")

=== Testing on known PLANET stars ===

KIC 4935189
  true:      P=9.3204d  depth=1977ppm
  recovered: P=9.3206d  depth=1317ppm
  period error: 0.00%  ✅ MATCH

KIC 7213876
  true:      P=61.7771d  depth=246ppm
  recovered: P=37.3432d  depth=258ppm
  period error: 39.55%  ❌ MISMATCH

KIC 9933041
  true:      P=5.5973d  depth=102ppm
  recovered: P=47.7299d  depth=719ppm
  period error: 752.74%  ❌ MISMATCH

=== Testing on a known EMPTY star (should find nothing convincing) ===

KIC 4845295 (no real planet)
  recovered: P=48.4133d  depth=114ppm  SDE=11.2
  (low SDE / low plausibility here is a good sign)


## 5. Calibrating the threshold

SDE alone does not tell you whether a planet is present. You need to know what
SDE the *empty* stars reach — that is your noise floor, and anything below it is
not a detection.

Run the search over a sample of dev stars, then compare the SDE distribution of
known-empty stars against known-planet stars. Where they overlap is where you
lose points, and narrowing that overlap is most of the work.

In [ ]:
N_SAMPLE = 20  # raise once you trust your pipeline

dev_files = sorted(glob.glob(f"{DEV_DIR}/*.parquet"))[:N_SAMPLE]
rows = []
for i, path in enumerate(dev_files, 1):
    t, f = clean(pd.read_parquet(path))
    r = {"star_id": os.path.basename(path)[:-8]} if t is None else \
        {**search(t, f), "star_id": os.path.basename(path)[:-8]}
    rows.append(r)
    print(f"  [{i}/{len(dev_files)}] {r['star_id']} SDE {r.get('sde', 0):.1f}", flush=True)

dev = pd.DataFrame(rows)

# attach ground truth
labels = pd.read_csv("dev_labels.csv")
dtruth = pd.read_csv("dev_truth.csv")
dev["kepid"] = dev.star_id.str.replace("KIC_", "").astype(int)
dev = dev.merge(labels[["kepid", "label"]], on="kepid", how="left")
dev = dev.merge(dtruth[["kepid", "injected", "bin", "period_days", "depth_ppm"]],
                on="kepid", how="left", suffixes=("", "_true"))
dev["injected"] = dev.injected.fillna(0)
dev["has_planet"] = ((dev.label == 1) | (dev.injected == 1)).astype(int)

print("\nSDE by truth:")
print(dev.groupby("has_planet").sde.describe()[["count", "50%", "max"]])

In [ ]:
empty = dev[dev.has_planet == 0].sde
planet = dev[dev.has_planet == 1].sde

print(f"{'SDE':>6} {'recall':>8} {'false pos':>10}")
print("-" * 26)
for thr in (6, 8, 10, 12, 15, 20):
    print(f"{thr:>6} {(planet > thr).mean():>8.0%} {(empty > thr).mean():>10.0%}")

# Pick the threshold that suits your strategy. Precision and recall are both
# scored, so neither extreme wins. Note that a single global cut is itself a
# weak choice -- SDE depends on stellar brightness and variability, so a
# per-star threshold, or a classifier over several features, will do better.
SDE_THRESHOLD = 10.0

In [ ]:
N_SAMPLE = 30  # number of dev stars to test on

dev_files = sorted(glob.glob(f"{DEV_DIR}/*.parquet"))[:N_SAMPLE]

labels = pd.read_csv("dev_labels.csv")
dtruth = pd.read_csv("dev_truth.csv")

rows = []
for i, path in enumerate(dev_files, 1):
    sid = os.path.basename(path)[:-8]
    df_star = pd.read_parquet(path)

    t_c, f_c = clean(df_star)
    if t_c is None:
        rows.append({"star_id": sid, "sde": 0.0, "plausibility": 0.0})
        continue

    r = search_smart(t_c, f_c)
    r["star_id"] = sid
    rows.append(r)
    print(f"  [{i}/{len(dev_files)}] {sid} SDE {r.get('sde', 0):.1f}", flush=True)

dev = pd.DataFrame(rows)

# attach ground truth (same logic as the original notebook's section 5)
dev["kepid"] = dev.star_id.str.replace("KIC_", "").astype(int)
dev = dev.merge(labels[["kepid", "label"]], on="kepid", how="left")
dev = dev.merge(dtruth[["kepid", "injected", "depth_ppm"]], on="kepid", how="left")
dev["injected"] = dev.injected.fillna(0)
dev["has_planet"] = ((dev.label == 1) | (dev.injected == 1)).astype(int)

print("\nSDE and plausibility by truth:")
print(dev.groupby("has_planet")[["sde", "plausibility"]].describe())

  [1/30] KIC_10064054 SDE 9.9
  [2/30] KIC_10262466 SDE 12.1
  [3/30] KIC_10294348 SDE 9.8
  [4/30] KIC_10319385 SDE 98.5
  [5/30] KIC_10385347 SDE 14.8
  [6/30] KIC_10406892 SDE 12.3
  [7/30] KIC_10453542 SDE 16.1
  [8/30] KIC_10717220 SDE 96.1
  [9/30] KIC_10718455 SDE 49.4
  [10/30] KIC_10919052 SDE 15.0
  [11/30] KIC_10928172 SDE 12.2
  [12/30] KIC_11068968 SDE 8.8
  [13/30] KIC_11253711 SDE 29.7
  [14/30] KIC_11457664 SDE 12.5
  [15/30] KIC_11496543 SDE 13.7
  [16/30] KIC_11751454 SDE 11.5
  [17/30] KIC_11773331 SDE 9.7
  [18/30] KIC_11774991 SDE 9.8
  [19/30] KIC_11804465 SDE 3774.1
  [20/30] KIC_11870928 SDE 8.9
  [21/30] KIC_12159489 SDE 10.2
  [22/30] KIC_12252612 SDE 12.7
  [23/30] KIC_12505525 SDE 12.9
  [24/30] KIC_12556897 SDE 19.9
  [25/30] KIC_12934451 SDE 14.1
  [26/30] KIC_2019199 SDE 12.7
  [27/30] KIC_2301053 SDE 15.7
  [28/30] KIC_2833811 SDE 24.0
  [29/30] KIC_2856200 SDE 483.4
  [30/30] KIC_2857607 SDE 12.5

SDE and plausibility by truth:
             sde         

In [ ]:
SDE_THRESHOLD = 11.0

def confidence_from_plausibility(plausibility, midpoint=280, steepness=0.003):
    """Squash plausibility into 0-1. Midpoint chosen from dev calibration."""
    return float(1.0 / (1.0 + np.exp(-steepness * (plausibility - midpoint))))

In [ ]:
!unzip -q private_pack.zip -d .

In [ ]:
print("private", len(glob.glob(f"{PRIVATE_DIR}/*.parquet")))

private 87


In [ ]:
import time

private_files = sorted(glob.glob(f"{PRIVATE_DIR}/*.parquet"))
print(f"Found {len(private_files)} private stars")

# Time just ONE star through the full pipeline
start = time.time()

test_path = private_files[0]
df_test = pd.read_parquet(test_path)
t_p, f_p = clean(df_test)
result_p = search_smart(t_p, f_p) if t_p is not None else {"sde": 0.0, "plausibility": 0.0}

elapsed = time.time() - start
print(f"\nStar: {os.path.basename(test_path)}")
print(f"Time for 1 star: {elapsed:.1f} seconds")
print(f"Estimated time for all {len(private_files)} stars: {elapsed * len(private_files) / 60:.1f} minutes")
print(f"\nResult: {result_p}")

Found 87 private stars

Star: STAR_0000.parquet
Time for 1 star: 19.4 seconds
Estimated time for all 87 stars: 28.1 minutes

Result: {'period': 28.236743919148463, 'depth_ppm': 233.68140525367966, 'duration_hours': 9.600000000000001, 't0': 508.55600374353514, 'sde': 9.701119381731095, 'plausibility': np.float64(209.9176030438289)}


## 6. Writing a submission

The `confidence` column drives the ranking metrics and is worth as much as the
hard `prediction`. Mapping SDE through a logistic is the laziest thing that
works; a properly calibrated probability will score better.

Timing note: the cell below runs the full private set. Time a few stars first
and multiply — do not start this at hour 70.

In [ ]:
def confidence_from_sde(s, midpoint=SDE_THRESHOLD, steepness=0.4):
    """Squash SDE into (0, 1). Replace with something calibrated."""
    return float(1.0 / (1.0 + np.exp(-steepness * (s - midpoint))))


def run_split(directory):
    out = []
    paths = sorted(glob.glob(f"{directory}/*.parquet"))
    for i, path in enumerate(paths, 1):
        sid = os.path.basename(path)[:-8]
        try:
            t, f = clean(pd.read_parquet(path))
            r = search(t, f) if t is not None else {"sde": 0.0}
        except Exception as e:
            print(f"  {sid} failed: {e}")
            r = {"sde": 0.0}

        s = r.get("sde", 0.0)
        hit = s > SDE_THRESHOLD
        out.append({
            "star_id": sid,
            "prediction": int(hit),
            "confidence": round(confidence_from_sde(s), 4),
            "period": round(r["period"], 5) if hit else None,
            "depth_ppm": round(r["depth_ppm"], 1) if hit else None,
            "duration_hours": round(r["duration_hours"], 3) if hit else None,
        })
        if i % 10 == 0:
            print(f"  [{i}/{len(paths)}]", flush=True)
    return pd.DataFrame(out)


sub = run_split(PRIVATE_DIR)
sub.to_csv("submission.csv", index=False)
print(f"\n{len(sub)} rows, {sub.prediction.sum()} detections")
sub.head()

In [ ]:
# Validate before submitting. Fix anything this prints.
s = pd.read_csv("submission.csv")
need = ["star_id", "prediction", "confidence", "period", "depth_ppm", "duration_hours"]

assert list(s.columns) == need, f"columns must be exactly {need}"
assert len(s) == 87, f"expected 87 rows, got {len(s)}"
assert s.star_id.nunique() == 87, "duplicate star_id"
assert s.star_id.str.match(r"^STAR_\d{4}$").all(), "bad star_id format"
assert s.prediction.isin([0, 1]).all(), "prediction must be 0 or 1"
assert s.confidence.between(0, 1).all(), "confidence out of range"

pos = s[s.prediction == 1]
for c in ("period", "depth_ppm", "duration_hours"):
    assert pos[c].notna().all(), f"{c} missing for some detections"

print(f"OK — {len(pos)} detections, {len(s) - len(pos)} non-detections")
print(f"confidence: {s.confidence.nunique()} unique values")

In [ ]:
def run_split_smart(directory):
    out = []
    paths = sorted(glob.glob(f"{directory}/*.parquet"))
    for i, path in enumerate(paths, 1):
        sid = os.path.basename(path)[:-8]
        try:
            t_s, f_s = clean(pd.read_parquet(path))
            r = search_smart(t_s, f_s) if t_s is not None else {"sde": 0.0, "plausibility": 0.0, "period": np.nan, "depth_ppm": np.nan, "duration_hours": np.nan}
        except Exception as e:
            print(f"  {sid} failed: {e}")
            r = {"sde": 0.0, "plausibility": 0.0, "period": np.nan, "depth_ppm": np.nan, "duration_hours": np.nan}

        plaus = r.get("plausibility", 0.0)
        hit = r.get("sde", 0.0) > SDE_THRESHOLD

        out.append({
            "star_id": sid,
            "prediction": int(hit),
            "confidence": round(confidence_from_plausibility(plaus), 4),
            "period": round(r["period"], 5) if hit else None,
            "depth_ppm": round(r["depth_ppm"], 1) if hit else None,
            "duration_hours": round(r["duration_hours"], 3) if hit else None,
        })
        if i % 10 == 0:
            print(f"  [{i}/{len(paths)}]", flush=True)
    return pd.DataFrame(out)


# Run on the full private set (~28 minutes, based on our timing test)
sub = run_split_smart(PRIVATE_DIR)
sub.to_csv("submission_spectralcode.csv", index=False)
print(f"\n{len(sub)} rows, {sub.prediction.sum()} detections")
sub.head()

  [10/87]
  [20/87]
  [30/87]
  [40/87]
  [50/87]
  [60/87]
  [70/87]
  [80/87]

87 rows, 46 detections


,star_id,prediction,confidence,period,depth_ppm,duration_hours
0,STAR_0000,0,0.4476,NaN,NaN,NaN
1,STAR_0001,1,0.5947,3.14154,33.9,19.2
2,STAR_0002,1,0.4729,3.11624,29.0,19.2
3,STAR_0003,0,0.4250,NaN,NaN,NaN
4,STAR_0004,0,0.4323,NaN,NaN,NaN


In [ ]:
# Validate before submitting — fix anything this prints
s = pd.read_csv("submission_spectralcode.csv")
need = ["star_id", "prediction", "confidence", "period", "depth_ppm", "duration_hours"]

assert list(s.columns) == need, f"columns must be exactly {need}"
assert len(s) == 87, f"expected 87 rows, got {len(s)}"
assert s.star_id.nunique() == 87, "duplicate star_id"
assert s.star_id.str.match(r"^STAR_\d{4}$").all(), "bad star_id format"
assert s.prediction.isin([0, 1]).all(), "prediction must be 0 or 1"
assert s.confidence.between(0, 1).all(), "confidence out of range"

pos = s[s.prediction == 1]
for c in ("period", "depth_ppm", "duration_hours"):
    assert pos[c].notna().all(), f"{c} missing for some detections"
assert (pos.period > 0).all(), "period must be positive"

print(f"OK — {len(pos)} detections, {len(s) - len(pos)} non-detections")
print(f"confidence: min {s.confidence.min():.3f}, max {s.confidence.max():.3f}, unique {s.confidence.nunique()}")

OK — 38 detections, 49 non-detections
confidence: min 0.150, max 1.000, unique 73


In [ ]:
detections = sub[sub.prediction == 1]
suspicious = detections[detections.duration_hours == 19.2]
print(f"Detections with duration = 19.2h (max grid value): {len(suspicious)} out of {len(detections)}")
print(suspicious[["star_id", "confidence", "period", "depth_ppm", "duration_hours"]].to_string())

Detections with duration = 19.2h (max grid value): 9 out of 46
      star_id  confidence    period  depth_ppm  duration_hours
1   STAR_0001      0.5947   3.14154       33.9            19.2
2   STAR_0002      0.4729   3.11624       29.0            19.2
11  STAR_0011      0.4648  30.73656      155.0            19.2
20  STAR_0020      0.6978   3.20090       83.5            19.2
33  STAR_0033      0.5547   3.14222       13.3            19.2
43  STAR_0043      0.5569   3.14245       40.2            19.2
48  STAR_0048      0.4375   3.11604       14.8            19.2
64  STAR_0064      0.4786   3.00328       24.6            19.2
71  STAR_0071      0.5173   3.14222       26.8            19.2


In [ ]:
# Flag any detections sitting suspiciously close to the ~3.1-day systematic period
SYSTEMATIC_PERIOD = 3.14
SYSTEMATIC_TOLERANCE = 0.15  # days

def is_systematic(period):
    if pd.isna(period):
        return False
    return abs(period - SYSTEMATIC_PERIOD) < SYSTEMATIC_TOLERANCE

sub["near_systematic"] = sub.period.apply(is_systematic)
print(f"Detections near systematic period: {sub.near_systematic.sum()}")
print(sub[sub.near_systematic][["star_id", "period", "depth_ppm", "confidence"]])

Detections near systematic period: 8
      star_id   period  depth_ppm  confidence
1   STAR_0001  3.14154       33.9      0.5947
2   STAR_0002  3.11624       29.0      0.4729
20  STAR_0020  3.20090       83.5      0.6978
33  STAR_0033  3.14222       13.3      0.5547
43  STAR_0043  3.14245       40.2      0.5569
48  STAR_0048  3.11604       14.8      0.4375
64  STAR_0064  3.00328       24.6      0.4786
71  STAR_0071  3.14222       26.8      0.5173


In [ ]:
# Correct the submission: these 8 are very likely a systematic artifact, not real planets
for idx in sub[sub.near_systematic].index:
    sub.loc[idx, "prediction"] = 0
    sub.loc[idx, "confidence"] = 0.15  # low confidence — likely nothing, but not zero certainty
    sub.loc[idx, "period"] = None
    sub.loc[idx, "depth_ppm"] = None
    sub.loc[idx, "duration_hours"] = None

sub = sub.drop(columns=["near_systematic"])  # remove helper column before saving

sub.to_csv("submission_spectralcode.csv", index=False)
print(f"Updated: {sub.prediction.sum()} detections, {len(sub) - sub.prediction.sum()} non-detections")

Updated: 38 detections, 49 non-detections


## Where to go from here

Every stage above is the weakest defensible version of itself.

**Detrending** is where the most points are. The running median is crude and its
window is a single global constant. Splines, Savitzky-Golay, Gaussian processes,
or iteratively masking in-transit points before fitting the trend all do better.
Long-duration transits of long-period planets are actively damaged by a 1-day
window.

**The search** uses a fixed coarse grid and refines a fixed number of peaks.
Coarse resolution sets your sensitivity floor. Consider matched filters, wavelet
methods, or transit-shaped kernels instead of boxes.

**The decision** is one global SDE cut. SDE depends on stellar brightness,
variability and transit duration. A classifier over several features — SDE, odd
vs even depth consistency, transit count, fitted depth, scatter — will separate
planets from noise far better than one number.

**Vetting** is absent entirely. Real pipelines reject candidates using checks
you can implement here: is the depth consistent between odd and even transits
(if not, it is likely an eclipsing binary at twice the period)? Is there a
secondary eclipse at phase 0.5? Does the signal appear in every quarter, or only
one? Does it sit at a known systematic period? Each false positive you remove is
precision you gain.

**Machine learning** is unused. You have a labelled training set. Folded light
curves as 1D inputs to a CNN is the standard approach in the literature. You
could also learn to rank BLS candidates rather than thresholding them.

**Confidence calibration** is a logistic with two made-up constants. Ranking
metrics are a large share of the score and this is cheap to improve.

One habit worth keeping: whenever your pipeline reports nothing, test it against
a known injected signal before concluding the data is empty. A silent failure
looks exactly like a clean star.